# Figures 5-8, 10 and 11

Figures of Sections 4.1-4.5, drawn from the per-cell metrics written by `run_inference.py` and by the scripts in
`experiments/`. Input locations are set at the top of each cell. Figures are written to `figures/output/` (TIFF at
600 dpi and PNG).

In [ ]:
# Data locations (environment variables, see README)
import os as _os, sys as _sys
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.getcwd(), '..')))
from landformer_paths import EXT_DIR, ROOT  # noqa: E402


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
OUT = os.path.join(ROOT, 'figures', 'output')
os.makedirs(OUT, exist_ok=True)

## Figure 5 - LandFormer and LSTM (Section 4.1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------------- DATA ----------------
fluxes = ['ET','SM','Q']

lstm = np.array([0.4813, 0.4335, 0.2856])
land = np.array([0.60,   0.59,   0.35])

y = np.arange(len(fluxes))

# ---------------- STYLE ----------------
plt.figure(figsize=(8.5,4.8))

lstm_color = '#e07a5f'      # muted coral
land_color = '#3d5a80'      # muted blue

# ----- performance zones -----
plt.axvspan(0, 0.4, color='#f2cc8f', alpha=0.25)   # poor
plt.axvspan(0.4, 0.6, color='#81b29a', alpha=0.20) # acceptable
plt.axvspan(0.6, 0.8, color='#2a9d8f', alpha=0.15) # good
plt.axvspan(-0.4, 0, color='#f2cc8f', alpha=0.25)
plt.axvspan(-0.6, -0.4, color='#81b29a', alpha=0.20)
plt.axvspan(-0.8, -0.6, color='#2a9d8f', alpha=0.15)

# bars
plt.barh(y, -lstm, height=0.25, color=lstm_color, edgecolor='none', label='LSTM')
plt.barh(y,  land, height=0.25, color=land_color, edgecolor='none', label='Landformer')

# center line
plt.axvline(0, color='black', linewidth=1)
plt.xlim(-0.8, 0.8)
# ticks
plt.yticks(y, fluxes, fontsize=13)
ticks = np.linspace(-0.8,0.8,9)
plt.xticks(ticks, [f"{abs(t):.1f}" for t in ticks], fontsize=11)

# labels
plt.xlabel('Median Kling–Gupta Efficiency (KGE)', fontsize=13)
#plt.title('Out-of-Sample Hydrological Skill', fontsize=15, weight='bold')

# annotate values
# for i in range(len(fluxes)):
#     plt.text(-lstm[i]-0.015, y[i], f"{lstm[i]:.2f}",
#              va='center', ha='right', fontsize=11)
#     plt.text( land[i]+0.015, y[i], f"{land[i]:.2f}",
#              va='center', ha='left', fontsize=11)

# cosmetics
ax = plt.gca()
ax.spines[['top','right','left','bottom']].set_visible(False)
plt.grid(axis='x', linestyle='--', alpha=0.25)

# model labels on top
plt.text(-0.6, 2.6, "LSTM", fontsize=12, weight='bold')
plt.text( 0.45, 2.6, "Landformer", fontsize=12, weight='bold')

# plt.legend(frameon=False, loc='lower right')
for _ext in ("tif", "png"):
    plt.savefig(f"{OUT}/Fig_5.{_ext}", bbox_inches="tight", facecolor="white", dpi=600 if _ext == "tif" else 200)
plt.show()


## Figure 6 - model capacity (Section 4.2)

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.gridspec import GridSpec

# -----------------------------
# STYLE SETTINGS (Clean + Large + Premium)
# -----------------------------
rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 17,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "axes.linewidth": 1.0,
    "axes.edgecolor": "black",
    "axes.grid": True,
    "grid.color": "#F8F3F3",
    "grid.linewidth": 0.8,
    "grid.linestyle": "-",
    "axes.axisbelow": True,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "figure.dpi": 600,
    "savefig.dpi": 600,
})

# -----------------------------
# File → parameter mapping
# -----------------------------
folder_path = f"{ROOT}/inference_output"

file_param_map = [
    ("grid_metrics_TEST_HYDRODL.csv",   "12.5M"),
    ("grid_metrics_TEST_HYDRODL_3.csv", "61.5M"),
    ("grid_metrics_TEST_HYDRODL_4.csv", "109M"),
]

files        = [os.path.join(folder_path, f) for f, _ in file_param_map]
model_labels = [lbl for _, lbl in file_param_map]
legend_labels = ["LandFormer-S (12.5M)", "LandFormer-B (61.5M)", "LandFormer-L (109M)"]
n_models     = len(files)

# -----------------------------
# Definitions
# -----------------------------
metrics = [
    ("_kge",         "KGE",         True),
    ("_nse",         "NSE",         True),
    ("_correlation", "Correlation", True),
    ("_rmse",        "RMSE",        False),
]

variables = [
    ("evap",                "Evapotranspiration (ET)", "#2166AC"),
    ("Q",                   "Total Runoff (Q)",           "#1A9850"),
    ("soil_moisture_total", "Soil Moisture (SM)",       "#D73027"),
]

n_metrics = len(metrics)
n_vars    = len(variables)

model_colors = ["#C0C0C0", "#7A7A7A", "#1A1A1A"]
model_sizes  = [120, 170, 230]

# -----------------------------
# Load medians
# -----------------------------
medians = []
for var_prefix, _, _ in variables:
    var_med = []
    for suffix, _, _ in metrics:
        col = f"{var_prefix}{suffix}"
        row = []
        for fpath in files:
            df  = pd.read_csv(fpath)
            arr = df[col].dropna().values \
                  if col in df.columns else np.array([np.nan])
            row.append(np.nanmedian(arr))
        var_med.append(np.array(row))
    medians.append(var_med)

# -----------------------------
# Figure Layout
# -----------------------------
panel_letters = "abcdefghijkl"
panel_idx     = 0

fig = plt.figure(figsize=(20, 13))
fig.patch.set_facecolor("white")

gs = GridSpec(n_vars, n_metrics, figure=fig,
              hspace=0.35, wspace=0.25,
              left=0.08, right=0.96,
              top=0.93, bottom=0.12)

for v_idx, (var_prefix, var_title, var_color) in enumerate(variables):
    for m_idx, (suffix, metric_label, higher_better) in enumerate(metrics):

        ax = fig.add_subplot(gs[v_idx, m_idx])
        ax.set_facecolor("white")

        vals  = medians[v_idx][m_idx].copy()
        y_pos = np.arange(n_models, dtype=float)

        # Dumbbell spine
        for yi in y_pos:
            ax.hlines(yi, vals.min(), vals.max(),
                      colors="#DDDDDD", linewidth=2.0, zorder=1)

        # Scatter points
        for model_i in range(n_models):
            ax.scatter(vals[model_i], y_pos[model_i],
                       s=model_sizes[model_i],
                       color=model_colors[model_i],
                       edgecolors="white",
                       linewidths=1.5,
                       zorder=3)

        # Delta annotation
        delta  = vals[-1] - vals[0]
        better = (delta > 0 and higher_better) or \
                 (delta < 0 and not higher_better)
        sign   = "+" if delta > 0 else ""
        dcol   = "#2E7D32" if better else "#C62828"

        ax.text(0.98, 0.02, f"Δ = {sign}{delta:.3f}",
                transform=ax.transAxes,
                ha="right", va="bottom",
                fontsize=14, fontweight="bold",
                color=dcol)

        # Y axis labels (left column only)
        ax.set_yticks(y_pos)
        if m_idx == 0:
            ax.set_yticklabels(model_labels, fontweight="bold")
        else:
            ax.set_yticklabels([])

        ax.set_ylim(-0.5, n_models - 0.5)
        ax.invert_yaxis()

        # X limits
        pad = (vals.max() - vals.min()) * 0.25
        ax.set_xlim(vals.min() - pad, vals.max() + pad)

        # X ticks
        ax.tick_params(axis="x", labelsize=14)

        if v_idx == n_vars - 1:
            ax.set_xlabel(metric_label, fontweight="bold", labelpad=8)
        else:
            ax.set_xlabel("")

        if v_idx == 0:
            ax.set_title(metric_label, fontweight="bold", pad=10)

        # Variable label on right
        if m_idx == n_metrics - 1:
            ax.text(1.08, 0.5, var_title,
                    transform=ax.transAxes,
                    ha="left", va="center",
                    fontsize=16, fontweight="bold",
                    color=var_color, rotation=90)

        # Panel letters
        ax.text(0.02, 0.95, f"({panel_letters[panel_idx]})",
                transform=ax.transAxes,
                fontsize=14, fontweight="bold",
                va="top")

        panel_idx += 1

# Shared Y label
fig.text(0.03, 0.50, "Model Size",
         ha="center", va="center",
         fontsize=18, fontweight="bold",
         rotation=90)

# Legend
legend_handles = [
    plt.scatter([], [], s=model_sizes[i],
                color=model_colors[i],
                edgecolors="white", linewidths=1.5,
                label=legend_labels[i])
    for i in range(n_models)
]

fig.legend(
    handles=legend_handles,
    title="Model (trainable parameters)",
    title_fontsize=16,
    fontsize=15,
    loc="lower center",
    ncol=n_models,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
)

for _ext in ("tif", "png"):
    plt.savefig(f"{OUT}/Fig_6.{_ext}", bbox_inches="tight", facecolor="white", dpi=600 if _ext == "tif" else 200)

plt.show()

## Figure 7 - training coverage (Section 4.3)

In [ ]:
from matplotlib import rcParams
# Metrics on the x-axis, training coverage as legend.
rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 16, "axes.titlesize": 18, "axes.labelsize": 17, "xtick.labelsize": 15, "ytick.labelsize": 14,
    "axes.linewidth": 1.2, "axes.edgecolor": "#111111", "axes.grid": True, "grid.color": "#E6E6E6",
    "grid.linewidth": 0.8, "axes.axisbelow": True, "xtick.direction": "out", "ytick.direction": "out",
    "figure.dpi": 600, "savefig.dpi": 600,
})
folder_path = f"{ROOT}/outputs/ensemble_coverage/ensemble_grid_metrics"
files       = sorted(glob.glob(os.path.join(folder_path, "grid_em_*.csv")))
total_grids = 104841
selected_em = np.linspace(0, len(files) - 1, 6, dtype=int)
train_labels = [f"{((idx+1)*2000/total_grids)*100:.1f}%" for idx in selected_em]
dfs = [pd.read_csv(files[idx]) for idx in selected_em]

metrics_top = [("_kge", "KGE"), ("_correlation", "Correlation"), ("_nse", "NSE")]
variables = [("evap", "Evapotranspiration", "#2C7FB8"), ("Q", "Total Runoff", "#1A9850"), ("soil_moisture_total", "Soil Moisture", "#D73027")]
cov_colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(selected_em)))
offsets = np.linspace(-0.32, 0.32, len(selected_em))

fig, axes = plt.subplots(2, 3, figsize=(20, 11.5), gridspec_kw=dict(height_ratios=[1.6, 1], hspace=0.30, wspace=0.20))
fig.patch.set_facecolor("white")
letters = iter("abcdef")
for j, (var, vname, vcol) in enumerate(variables):
    ax = axes[0, j]; lo = 0.0
    for m, (suf, mlab) in enumerate(metrics_top):
        if m % 2 == 0:
            ax.axvspan(m - 0.5, m + 0.5, color="#F8F8F8", zorder=0)
        for k, df in enumerate(dfs):
            a = df[var + suf].dropna().clip(lower=-1).values
            q25, q50, q75 = np.percentile(a, [25, 50, 75]); lo = min(lo, q25)
            ax.plot([m + offsets[k]] * 2, [q25, q75], color=cov_colors[k], lw=3.2, solid_capstyle="round", zorder=4)
            ax.scatter(m + offsets[k], q50, s=70, color=cov_colors[k], edgecolor="white", linewidths=1.4, zorder=5,
                       label=train_labels[k] if (j == 0 and m == 0) else None)
    ax.set_xticks(range(len(metrics_top))); ax.set_xticklabels([l for _, l in metrics_top], fontweight="bold")
    ax.set_xlim(-0.5, len(metrics_top) - 0.5); ax.set_ylim(min(0, np.floor((lo - 0.05) * 10) / 10), 1.02)
    ax.set_title(vname, fontweight="bold", color=vcol, pad=10)
    if j == 0: ax.set_ylabel("Metric value", fontweight="bold")
    ax.text(0.02, 1.02, f"({next(letters)})", transform=ax.transAxes, fontsize=16, fontweight="bold", va="bottom")
for j, (var, vname, vcol) in enumerate(variables):
    ax = axes[1, j]
    for k, df in enumerate(dfs):
        a = df[var + "_rmse"].dropna().values
        q25, q50, q75 = np.percentile(a, [25, 50, 75])
        ax.plot([offsets[k]] * 2, [q25, q75], color=cov_colors[k], lw=3.2, solid_capstyle="round", zorder=4)
        ax.scatter(offsets[k], q50, s=70, color=cov_colors[k], edgecolor="white", linewidths=1.4, zorder=5)
    ax.set_xticks([0]); ax.set_xticklabels(["RMSE"], fontweight="bold"); ax.set_xlim(-0.5, 0.5)
    unit = "m$^3$ m$^{-3}$" if var == "soil_moisture_total" else "mm"
    ax.set_ylabel(f"RMSE ({unit})", fontweight="bold")
    ax.text(0.02, 1.02, f"({next(letters)})", transform=ax.transAxes, fontsize=16, fontweight="bold", va="bottom")
for ax in axes.flat:
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
fig.legend(*axes[0, 0].get_legend_handles_labels(), title="Training Data Coverage (%)", title_fontsize=16,
           loc="lower center", ncol=len(selected_em), fontsize=15, frameon=False, bbox_to_anchor=(0.5, -0.03))
for _ext in ("tif", "png"):
    plt.savefig(f"{OUT}/Fig_7.{_ext}", bbox_inches="tight", facecolor="white", dpi=600 if _ext == "tif" else 200)
plt.show()


## Figure 8 - autoregressive inputs (Section 4.4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib import rcParams
from matplotlib.gridspec import GridSpec

# -------------------------------------------------
# PREMIUM STYLE (Bigger + Cleaner + Strong Contrast)
# -------------------------------------------------
rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 17,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "axes.linewidth": 1.2,
    "axes.edgecolor": "#111111",
    "axes.grid": False,
    "figure.dpi": 600,
    "savefig.dpi": 600,
})

# -----------------------------
# Paths (unchanged)
# -----------------------------
baseline_file = f"{ROOT}/outputs/ensemble_coverage/ensemble_grid_metrics/grid_em_37.csv"

lagged_files = {
    "LF-AR-ET":  f"{ROOT}/inference_output/grid_metrics_Lagged_E.csv",
    "LF-AR-Q":   f"{ROOT}/inference_output/grid_metrics_Lagged_Q.csv",
    "LF-AR-SM":  f"{ROOT}/inference_output/grid_metrics_Lagged_SM.csv",
    "LF-AR-All": f"{ROOT}/inference_output/grid_metrics_Large_full_lag.csv",
}

# -----------------------------
# Definitions
# -----------------------------
metrics = [
    ("_kge",         "KGE",         True),
    ("_nse",         "NSE",         True),
    ("_correlation", "Correlation", True),
    ("_rmse",        "RMSE",        False),
]

variables = [
    ("evap",                "ET"),
    ("Q",                   "Q"),
    ("soil_moisture_total", "SM"),
]

ar_names  = list(lagged_files.keys())
ar_colors = ["#1F77B4", "#2CA02C", "#D62728", "#9467BD"]
ar_files  = list(lagged_files.values())
n_ar      = len(ar_names)

model_files  = [baseline_file] + ar_files

# -----------------------------
# Load medians
# -----------------------------
medians = []
for var_prefix, var_lbl in variables:
    var_med = []
    for suffix, _, _ in metrics:
        col = f"{var_prefix}{suffix}"
        model_meds = []
        for fpath in model_files:
            df  = pd.read_csv(fpath)
            arr = df[col].dropna().values if col in df.columns else np.array([np.nan])
            model_meds.append(np.nanmedian(arr))
        var_med.append(model_meds)
    medians.append(var_med)

# -----------------------------
# Compute deltas
# -----------------------------
deltas = []
for v_idx in range(len(variables)):
    var_delta = []
    for m_idx, (suffix, _, higher_better) in enumerate(metrics):
        baseline_val = medians[v_idx][m_idx][0]
        d = []
        for i in range(1, n_ar + 1):
            raw_delta = medians[v_idx][m_idx][i] - baseline_val
            if not higher_better:
                raw_delta = -raw_delta
            d.append(raw_delta)
        var_delta.append(d)
    deltas.append(var_delta)

# -----------------------------
# Figure Layout
# -----------------------------
panel_letters = "abcdefghijkl"
panel_idx     = 0

fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor("white")

gs = GridSpec(3, 4, figure=fig,
              hspace=0.60, wspace=0.35,
              left=0.08, right=0.95,
              top=0.94, bottom=0.15)

x     = np.arange(n_ar)
width = 0.65

for v_idx, (_, var_lbl) in enumerate(variables):
    for m_idx, (_, metric_lbl, _) in enumerate(metrics):

        ax = fig.add_subplot(gs[v_idx, m_idx])
        ax.set_facecolor("white")

        d_vals = deltas[v_idx][m_idx]

        # Strong zero reference
        ax.axhline(0, color="black", linewidth=1.4, linestyle="-", alpha=0.6)

        # Bars
        for i in range(n_ar):
            ax.bar(x[i], d_vals[i],
                   width=width,
                   color=ar_colors[i],
                   edgecolor="black",
                   linewidth=1.2,
                   alpha=0.85,
                   zorder=3)

            # Value label
            sign  = "+" if d_vals[i] >= 0 else ""
            offset = max(abs(np.array(d_vals))) * 0.05
            ytext = d_vals[i] + offset if d_vals[i] >= 0 else d_vals[i] - offset
            va    = "bottom" if d_vals[i] >= 0 else "top"

            ax.text(x[i], ytext,
                    f"{sign}{d_vals[i]:.3f}",
                    ha="center", va=va,
                    fontsize=12, fontweight="bold")

        # Dynamic Y limits
        d_range = max(d_vals) - min(d_vals)
        pad     = d_range * 0.35 if d_range > 0 else 0.08
        ax.set_ylim(min(0, min(d_vals)) - pad,
                    max(0, max(d_vals)) + pad)

        # X ticks
        ax.set_xticks(x)
        if v_idx == 2:
            ax.set_xticklabels(ar_names,
                               rotation=30,
                               ha="right",
                               fontsize=13)
        else:
            ax.set_xticklabels([])

        # Titles
        if v_idx == 0:
            ax.set_title(f"Δ {metric_lbl}",
                         fontsize=18,
                         fontweight="bold",
                         pad=12)

        # Variable label on right
        if m_idx == 3:
            ax.text(1.08, 0.5, var_lbl,
                    transform=ax.transAxes,
                    ha="left", va="center",
                    fontsize=18,
                    fontweight="bold",
                    rotation=270)

        # Clean spines
        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)
        for spine in ["bottom", "left"]:
            ax.spines[spine].set_linewidth(1.0)

        # Panel letter
        ax.text(0.03, 1.02,
                f"({panel_letters[panel_idx]})",
                transform=ax.transAxes,
                fontsize=15,
                fontweight="bold",
                va="top")

        panel_idx += 1

# -----------------------------
# Legend
# -----------------------------
legend_handles = [
    mpatches.Patch(color=ar_colors[i], label=ar_names[i])
    for i in range(n_ar)
]
legend_handles.append(
    mlines.Line2D([], [], color="black", linewidth=1.4,
                  label="LandFormer-NAR (Reference)")
)

fig.legend(handles=legend_handles,
           loc="lower center",
           ncol=n_ar + 1,
           fontsize=14,
           frameon=False,
           bbox_to_anchor=(0.5, 0.04))

fig.text(0.02, 0.50,
         "Improvement over LandFormer-NAR",
         ha="center", va="center",
         fontsize=18, fontweight="bold",
         rotation=90)

for _ext in ("tif", "png"):
    plt.savefig(f"{OUT}/Fig_8.{_ext}", bbox_inches="tight", facecolor="white", dpi=600 if _ext == "tif" else 200)

plt.show()

## Figure 10 - training resolution, per-cell distributions (Section 4.5)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams
import warnings
warnings.filterwarnings("ignore")

# =========================================================
# PREMIUM STYLE
# =========================================================
rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.2,
    "axes.edgecolor": "#111111",
    "figure.dpi": 600,
    "savefig.dpi": 600,
})

# =========================================================
# DATA (UNCHANGED PATHS)
# =========================================================
baseline_path = f"{ROOT}/inference_output/grid_metrics_Large_full_lag.csv"

model_paths = {
    "LF-AR-All":   baseline_path,
    "LF-AR-1°":    f"{ROOT}/outputs/ensemble_resolution/individual_grid_metrics/model_1_grid_metrics.csv",
    "LF-AR-0.5°":  f"{ROOT}/outputs/ensemble_resolution/individual_grid_metrics/model_2_grid_metrics.csv",
    "LF-AR-0.25°": f"{ROOT}/outputs/ensemble_resolution/individual_grid_metrics/model_3_grid_metrics.csv",
    "EM(1°+0.5°)": f"{ROOT}/outputs/ensemble_resolution/ensemble_grid_metrics/grid_em_2.csv",
    "EM(All)":     f"{ROOT}/outputs/ensemble_resolution/ensemble_grid_metrics/grid_em_3.csv",
}

model_names = list(model_paths.keys())
model_files = list(model_paths.values())
n_models    = len(model_names)

variables = [
    ("evap",                "Evapotranspiration (ET)", "#1F4E79"),
    ("Q",                   "Total Runoff (Q)",        "#1A9850"),
    ("soil_moisture_total", "Soil Moisture (SM)",      "#B22222"),
]

metrics = [
    ("_kge",         "KGE"),
    ("_nse",         "NSE"),
    ("_correlation", "Correlation"),
    ("_rmse",        "RMSE"),
]

dfs = [pd.read_csv(f) for f in model_files]

# =========================================================
# COLORS (Stronger contrast)
# =========================================================
box_colors = [
    "#2C3E50",  # baseline
    "#E74C3C",
    "#E67E22",
    "#F39C12",
    "#8E44AD",
    "#16A085",
]

model_types = ["baseline", "individual", "individual",
               "individual", "ensemble", "ensemble"]

# =========================================================
# FIGURE
# =========================================================
fig, axes = plt.subplots(3, 4,
                         figsize=(20, 14),
                         gridspec_kw={"hspace": 0.20,
                                      "wspace": 0.30})
fig.patch.set_facecolor("white")

for v_idx, (vpfx, vlbl, vcol) in enumerate(variables):
    for mi, (sfx, mlbl) in enumerate(metrics):

        ax = axes[v_idx, mi]
        col = f"{vpfx}{sfx}"

        data_to_plot = []
        medians = []

        for df in dfs:
            arr = df[col].dropna().values \
                  if col in df.columns else np.array([np.nan])
            p2, p98 = np.nanpercentile(arr, 2), np.nanpercentile(arr, 98)
            arr_c = arr[(arr >= p2) & (arr <= p98)]
            data_to_plot.append(arr_c)
            medians.append(np.nanmedian(arr))

        bp = ax.boxplot(
            data_to_plot,
            patch_artist=True,
            showfliers=False,
            widths=0.6,
            medianprops=dict(color="white",
                             linewidth=2.8),
            whiskerprops=dict(linewidth=1.3,
                              color="#555555"),
            capprops=dict(linewidth=1.5),
            boxprops=dict(linewidth=1.2),
        )

        # Apply colors
        for patch, color, mtype in zip(bp["boxes"],
                                       box_colors,
                                       model_types):
            patch.set_facecolor(color)
            patch.set_alpha(0.95 if mtype == "baseline" else 0.85)
            patch.set_edgecolor("#222222")
            if mtype == "ensemble":
                patch.set_hatch("///")

        # Baseline median line
        ax.axhline(medians[0],
                   color="#2C3E50",
                   linewidth=1.8,
                   linestyle="--",
                   alpha=0.8)

        # Ensemble region shading
        ax.axvspan(4.5, 6.5,
                   color="#F3E5F5",
                   alpha=0.5,
                   zorder=0)

        # Median diamond
        for i, med in enumerate(medians):
            ax.plot(i + 1, med,
                    marker="D",
                    color="white",
                    markersize=6,
                    markeredgecolor="#111111",
                    markeredgewidth=1.2,
                    zorder=6)

        # Panel labels
        panel_label = chr(97 + v_idx * 4 + mi)
        ax.text(-0.02, 1.03,                      # panel letters above the axes
                f"({panel_label})",
                transform=ax.transAxes,
                fontsize=14,
                fontweight="bold",
                ha="right", va="bottom")

        # Titles
        if v_idx == 0:
            ax.set_title(mlbl,
                         fontsize=18,
                         fontweight="bold",
                         pad=12)

        # Y-axis labels
        if mi == 0:
            ax.set_ylabel(vlbl,
                          fontsize=18,
                          fontweight="bold",
                          color=vcol,
                          labelpad=12)

        # X-axis ticks
        ax.set_xticks(range(1, n_models + 1))
        if v_idx == 2:
            ax.set_xticklabels(model_names,
                               fontsize=14,
                               fontweight="bold",
                               rotation=25,
                               ha="right")
        else:
            ax.set_xticklabels([])

        # Separator line
        ax.axvline(4.5,
                   color="#888888",
                   linewidth=1.5,
                   linestyle=":")

        # Bold ticks
        ax.tick_params(axis="both",
                       labelsize=14,
                       width=1.2)

        # Stronger spines
        for spine in ax.spines.values():
            spine.set_linewidth(1.2)
            spine.set_color("#111111")

        ax.set_xlim(0.3, n_models + 0.7)
        ax.grid(axis="y",
                linewidth=0.6,
                color="#DDDDDD")

# =========================================================
# LEGEND
# =========================================================
legend_patches = [
    mpatches.Patch(facecolor=box_colors[i],
                   edgecolor="#222222",
                   hatch="///" if model_types[i]=="ensemble" else "",
                   label=model_names[i])
    for i in range(n_models)
]

fig.legend(handles=legend_patches,
           loc="lower center",
           ncol=n_models,
           fontsize=14,
           frameon=False,
           bbox_to_anchor=(0.5, -0.01))

plt.tight_layout(rect=[0, 0.07, 1, 0.98])

for _ext in ("tif", "png"):
    plt.savefig(f"{OUT}/Fig_10.{_ext}", bbox_inches="tight", facecolor="white", dpi=600 if _ext == "tif" else 200)

plt.show()

## Figure 11 - training resolution, retention relative to LandFormer-AR-All (Section 4.5)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib import rcParams
import matplotlib.cm as cm
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings("ignore")

rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman"],
    "font.size":         18,
    "figure.dpi":        1200,
    "savefig.dpi":       600,
    "axes.linewidth":    0.8,
    "xtick.major.size":  0,
    "ytick.major.size":  0,
})

# =========================================================
# PATHS
# =========================================================
baseline_path = f"{ROOT}/inference_output/grid_metrics_Large_full_lag.csv"

model_paths = {
    "LF-AR-1°":    f"{ROOT}/outputs/ensemble_resolution/individual_grid_metrics/model_1_grid_metrics.csv",
    "LF-AR-0.5°":  f"{ROOT}/outputs/ensemble_resolution/individual_grid_metrics/model_2_grid_metrics.csv",
    "LF-AR-0.25°": f"{ROOT}/outputs/ensemble_resolution/individual_grid_metrics/model_3_grid_metrics.csv",
    "EM (1°+0.5°)": f"{ROOT}/outputs/ensemble_resolution/ensemble_grid_metrics/grid_em_2.csv",
    "EM (All)":     f"{ROOT}/outputs/ensemble_resolution/ensemble_grid_metrics/grid_em_3.csv",
}

model_names = list(model_paths.keys())
n_models    = len(model_names)
n_ind       = 3
n_ens       = 2

variables = [
    ("evap",                "ET", "Evapotranspiration (ET)", "#2166AC"),
    ("Q",                   "Q",  "Total Runoff (Q)",        "#1A9850"),
    ("soil_moisture_total", "SM", "Soil Moisture (SM)",      "#D6604D"),
]

metrics = [
    ("_kge",         "KGE",         True),
    ("_nse",         "NSE",         True),
    ("_correlation", "Corr.",       True),
    ("_rmse",        "RMSE",        False),
]

n_vars    = len(variables)
n_metrics = len(metrics)
n_cols    = n_vars * n_metrics   # 12

# =========================================================
# LOAD AND COMPUTE
# =========================================================
df_base   = pd.read_csv(baseline_path)
model_dfs = [pd.read_csv(f) for f in model_paths.values()]

retain_mat = np.full((n_models, n_cols), np.nan)
abs_mat    = np.full((n_models, n_cols), np.nan)
base_vals  = np.full(n_cols, np.nan)

metric_labels = []
var_idx_map   = []

c = 0
for v_idx, (vpfx, vshort, vlbl, vcol) in enumerate(variables):
    for sfx, mlbl, higher in metrics:
        col      = f"{vpfx}{sfx}"
        base_arr = df_base[col].dropna().values \
                   if col in df_base.columns else np.array([np.nan])
        base_med = np.nanmedian(base_arr)
        base_vals[c] = base_med
        metric_labels.append(mlbl)
        var_idx_map.append(v_idx)

        for m_idx, df_c in enumerate(model_dfs):
            comp_arr = df_c[col].dropna().values \
                       if col in df_c.columns else np.array([np.nan])
            comp_med = np.nanmedian(comp_arr)
            abs_mat[m_idx, c] = comp_med

            if higher:
                r = (comp_med / base_med * 100) \
                    if base_med > 0 else np.nan
            else:
                r = (base_med / comp_med * 100) \
                    if comp_med > 0 else np.nan
            retain_mat[m_idx, c] = r
        c += 1

# =========================================================
# COLORMAP
# =========================================================
cmap = LinearSegmentedColormap.from_list(
    "ret",
    ["#67001F", "#B2182B", "#D6604D",
     "#F4A582", "#FDDBC7",
     "#FFFFFF",
     "#D1E5F0", "#92C5DE",
     "#4393C3", "#2166AC", "#053061"],
    N=512
)
cmap.set_bad("lightgrey")

vmin = max(58, np.nanmin(retain_mat) - 2)
vmax = min(112, np.nanmax(retain_mat) + 2)
norm = TwoSlopeNorm(vmin=vmin, vcenter=100, vmax=vmax)

var_colors = [v[3] for v in variables]

# =========================================================
# FIGURE
# Precise layout using absolute axes positioning
# =========================================================
fig_w = 20
fig_h = 10
fig   = plt.figure(figsize=(fig_w, fig_h), facecolor="white")

# Layout constants (in figure fraction)
left_margin   = 0.115
right_margin  = 0.895
cbar_left     = 0.910
cbar_width    = 0.012
top_margin    = 0.88
bot_margin    = 0.12
gap           = 0.15   # gap between ind and ens panels

total_h       = top_margin - bot_margin
ind_frac      = n_ind / (n_ind + n_ens)
ens_frac      = n_ens / (n_ind + n_ens)
ind_h         = (total_h - gap) * (n_ind / (n_ind + n_ens + 0.3))
ens_h         = (total_h - gap) * (n_ens / (n_ind + n_ens + 0.3))
panel_w       = right_margin - left_margin

ind_bottom    = bot_margin + ens_h + gap
ens_bottom    = bot_margin

ax_ind = fig.add_axes([left_margin, ind_bottom, panel_w, ind_h])
ax_ens = fig.add_axes([left_margin, ens_bottom, panel_w, ens_h])

# =========================================================
# DRAW FUNCTION
# =========================================================
def draw_panel(ax, data, abs_data, row_names,
               show_var_labels, is_ensemble):

    n_r, n_c = data.shape

    im = ax.imshow(
        data,
        cmap=cmap,
        norm=norm,
        aspect="auto",
        interpolation="nearest",
        extent=[-0.5, n_c - 0.5, n_r - 0.5, -0.5]
    )

    # ── Variable group background shading ──────────────
    var_bg = ["#F7FBFF", "#F0FAF0", "#FFF5F0"]
    for v_idx in range(n_vars):
        x0 = v_idx * n_metrics - 0.5
        ax.axvspan(x0, x0 + n_metrics,
                   color=var_bg[v_idx],
                   alpha=0.12, zorder=0)

    # ── Cell border lines ───────────────────────────────
    for r in range(n_r + 1):
        lw   = 2.5 if r in [0, n_r] else 1.2
        col  = "#AAAAAA" if r in [0, n_r] else "white"
        ax.axhline(r - 0.5, color=col,
                   linewidth=lw, zorder=4)

    for c_i in range(n_c + 1):
        # Thick separator between variable groups
        if c_i in [0, n_metrics, 2 * n_metrics, n_c]:
            ax.axvline(c_i - 0.5,
                       color="#444444",
                       linewidth=2.0, zorder=5)
        else:
            ax.axvline(c_i - 0.5,
                       color="white",
                       linewidth=0.8, zorder=4)

    # ── Cell text ───────────────────────────────────────
    for r in range(n_r):
        for c_i in range(n_c):
            val = data[r, c_i]
            if np.isnan(val):
                continue
            dev = abs(val - 100)
            tc  = "white" if dev > 18 else "#1A1A1A"

            # Retention %
            ax.text(c_i, r - 0.13,
                    f"{val:.1f}%",
                    ha="center", va="center",
                    fontsize=18,
                    fontweight="bold",
                    color=tc,
                    zorder=6)

            # Absolute value
            abs_v = abs_data[r, c_i]
            ax.text(c_i, r + 0.27,
                    f"({abs_v:.3f})",
                    ha="center", va="center",
                    fontsize=14,
                    color=tc,
                    alpha=0.9,
                    zorder=6)

    # ── Y axis ─────────────────────────────────────────
    ax.set_yticks(range(n_r))
    ax.set_yticklabels(row_names,
                       fontsize=16,
                       fontweight="bold",
                       color="#8B44AD" if is_ensemble \
                             else "#222222")
    ax.yaxis.set_tick_params(pad=6)

    # ── X axis ─────────────────────────────────────────
    ax.set_xticks(range(n_c))
    ax.set_xticklabels(metric_labels,
                       fontsize=16,
                       color="#333333")
    ax.xaxis.set_tick_params(pad=4)

    # ── Axis limits ────────────────────────────────────
    ax.set_xlim(-0.5, n_c - 0.5)
    ax.set_ylim(n_r - 0.5, -0.5)

    # ── Spines ─────────────────────────────────────────
    ec = "#8B44AD" if is_ensemble else "#888888"
    lw = 1.5 if is_ensemble else 1.0
    for spine in ax.spines.values():
        spine.set_edgecolor(ec)
        spine.set_linewidth(lw)

    return im

# Draw both panels
im = draw_panel(ax_ind,
                retain_mat[:n_ind, :],
                abs_mat[:n_ind, :],
                model_names[:n_ind],
                show_var_labels=True,
                is_ensemble=False)

draw_panel(ax_ens,
           retain_mat[n_ind:, :],
           abs_mat[n_ind:, :],
           model_names[n_ind:],
           show_var_labels=False,
           is_ensemble=True)

# =========================================================
# VARIABLE GROUP LABELS — centred over each group of 4 cols
# placed between the two panels using figure coordinates
# =========================================================
col_width = panel_w / n_cols

for v_idx, (_, vshort, vlbl, vcol) in enumerate(variables):
    # Centre x in figure fraction
    col_start   = left_margin + v_idx * n_metrics * col_width
    col_centre  = col_start + (n_metrics * col_width) / 2

    # Top of ind panel
    fig.text(col_centre,
             ind_bottom + ind_h + 0.012,
             vlbl,
             ha="center", va="bottom",
             fontsize=10.5, fontweight="bold",
             color=vcol,
             transform=fig.transFigure)

    # Underline bar
    fig.add_artist(
        plt.Line2D(
            [col_start + 0.003,
             col_start + n_metrics * col_width - 0.003],
            [ind_bottom + ind_h + 0.008,
             ind_bottom + ind_h + 0.008],
            transform=fig.transFigure,
            color=vcol, linewidth=2.5,
            solid_capstyle="round"
        )
    )

# =========================================================
# BASELINE ROW — between the two panels
# =========================================================
mid_y = (ind_bottom + ens_bottom + ens_h) / 2 - 0.01

fig.text(left_margin - 0.005,
         mid_y,
         "Baseline (LF-AR-All):",
         ha="right", va="center",
         fontsize=14, fontweight="bold",
         color="#2C3E50",
         transform=fig.transFigure)

for c_i in range(n_cols):
    v_idx = var_idx_map[c_i]
    x_pos = left_margin + (c_i + 0.5) * col_width
    fig.text(x_pos, mid_y,
             f"{base_vals[c_i]:.3f}",
             ha="center", va="center",
             fontsize=14,
             color=var_colors[v_idx],
             fontweight="bold",
             transform=fig.transFigure)

# Separator line above and below baseline row
for y_line in [ind_bottom - 0.005, ens_bottom + ens_h + 0.005]:
    fig.add_artist(
        plt.Line2D(
            [left_margin, right_margin],
            [y_line, y_line],
            transform=fig.transFigure,
            color="#CCCCCC",
            linewidth=0.8,
            linestyle="--"
        )
    )

# =========================================================
# PANEL SECTION LABELS
# =========================================================
fig.text(left_margin,
         ind_bottom + ind_h + 0.058,
         "Individual Coarse-Resolution Models",
         ha="left", va="bottom",
         fontsize=14, color="#444444",
         fontstyle="italic",
         fontweight="bold",
         transform=fig.transFigure)

fig.text(left_margin,
         ens_bottom + ens_h + 0.008,
         "Multi-Resolution Ensembles",
         ha="left", va="bottom",
         fontsize=14, color="#8B44AD",
         fontstyle="italic",
         fontweight="bold",
         transform=fig.transFigure)

# =========================================================
# COLORBAR — precisely aligned to full panel height
# =========================================================
cbar_ax = fig.add_axes([cbar_left, bot_margin,
                         cbar_width,
                         top_margin - bot_margin])
cbar    = fig.colorbar(
    cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cbar_ax,
    extend="both"
)
cbar.set_label("Retention (%)",
               fontsize=14, labelpad=10)
cbar.ax.tick_params(labelsize=10)
cbar.outline.set_linewidth(0.8)

# 100% line
frac = (100 - vmin) / (vmax - vmin)
cbar.ax.axhline(frac,
                color="#111111",
                linewidth=1.5,
                linestyle="--",
                zorder=5)
cbar.ax.text(1.5, frac,
             "> 100%",
             transform=cbar.ax.transAxes,
             ha="left", va="center",
             fontsize=12, color="#111111")




for _ext in ("tif", "png"):
    plt.savefig(f"{OUT}/Fig_11.{_ext}", bbox_inches="tight", facecolor="white", dpi=600 if _ext == "tif" else 200)
plt.show()
